# Phase 5 — Model Training, Evaluation and Selection

The full training run lives in `src/train.py` (five candidates, 5-fold CV, SMOTE
comparison, randomised tuning, threshold selection). This notebook loads the
artifacts that run produced and interrogates the result.

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd, numpy as np
pd.set_option('display.width', 120)

import json
from config import MODELS_DIR, REPORTS_DIR
meta = json.loads((MODELS_DIR/'model_metadata.json').read_text())
print('final model:', meta['final_model'])
print('params    :', meta['best_params'])
print('threshold :', meta['decision_threshold'])

final model: Random Forest
params    : {'clf__n_estimators': 500, 'clf__min_samples_leaf': 10, 'clf__max_features': 'sqrt', 'clf__max_depth': 8}
threshold : 0.31


## Model comparison

In [2]:
pd.read_csv(REPORTS_DIR/'model_comparison.csv')

,model,cv_roc_auc,cv_recall,cv_precision,cv_f1,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tuned
0,Logistic Regression,0.8463,0.7873,0.5230,0.6282,0.5,0.7431,0.5106,0.7754,0.6157,0.8436,0.6510,True
1,XGBoost,0.8395,0.7445,0.5403,0.6260,0.5,0.7559,0.5279,0.7594,0.6228,0.8372,0.6497,False
2,Random Forest,0.8419,0.6849,0.5870,0.6321,0.5,0.7750,0.5646,0.6658,0.6110,0.8368,0.6436,True
3,SVM (RBF),0.8245,0.7572,0.5325,0.6251,0.5,0.7779,0.5731,0.6390,0.6043,0.8246,0.6058,False
4,KNN,0.8215,0.5204,0.6248,0.5675,0.5,0.7807,0.6019,0.5134,0.5541,0.8142,0.5895,False


Logistic Regression is a genuinely strong baseline here (test AUC 0.844) — churn in
this dataset is close to linearly separable in the engineered space. The tuned Random
Forest wins on cross-validated AUC (0.8479) and is chosen, but the margin over a
well-regularised linear model is small, and that is worth saying out loud rather than
hiding behind the winner.

## Why recall, not accuracy

In [3]:
d, b = meta['metrics_at_default_threshold'], meta['metrics_at_business_threshold']
print(f"{'':<12}{'t=0.50':>10}{'t=' + str(b['threshold']):>10}")
for k in ['accuracy', 'precision', 'recall', 'f1']:
    print(f'{k:<12}{d[k]:>10.3f}{b[k]:>10.3f}')
print()
print('confusion at business threshold:', b['confusion_matrix'])

                t=0.50    t=0.31
accuracy         0.764     0.678
precision        0.539     0.447
recall           0.783     0.896
f1               0.638     0.596

confusion at business threshold: {'tn': 620, 'fp': 415, 'fn': 39, 'tp': 335}


Moving the threshold from 0.50 to 0.31 costs 8.7 points of accuracy and buys
**11 points of recall** — false negatives drop from 81 to 39. In business terms:
42 additional churners caught, at the price of 164 extra retention offers. At $35
an offer and roughly $893 of annual revenue per churner, that trade is strongly
positive, which is exactly what the threshold search optimised.

## The economics behind the operating point

In [4]:
e = meta['economics']
for k, v in e.items():
    print(f'{k:<38}: {v}')

retention_offer_cost                  : 35.0
assumed_save_rate                     : 0.3
horizon_months                        : 12
test_set_annual_value_default         : 59304.46
test_set_annual_value_business        : 63690.78
projected_annual_value_500k_book      : 22601412.35


In [5]:
curve = pd.read_csv(REPORTS_DIR/'threshold_curve.csv')
best = curve.loc[curve.expected_profit.idxmax()]
print('profit-maximising threshold on cross-validated train predictions:')
print(best.round(3).to_string())

profit-maximising threshold on cross-validated train predictions:
threshold               0.310
flagged              3049.000
recall                  0.906
precision               0.444
f1                      0.596
expected_profit    264775.120


The threshold is selected on **cross-validated training predictions**, never on the
test set. Picking it on test would tune a hyperparameter to the held-out data and
make the reported recall optimistic — the same class of error as fitting a scaler on
test.